In [0]:
%sql
CREATE OR REPLACE TABLE rideflow.gold.fact_trips
CLUSTER BY (trip_date_ist, city_id)
AS
SELECT
  t.trip_id,
  t.trip_date_ist,
  t.request_ts,
  t.city_id,
  c.city_name,
  t.rider_id,
  t.driver_id,
  d.driver_sk                    AS driver_sk,     
  t.vehicle_type,
  t.trip_status,
  t.trip_status = 'COMPLETED'    AS is_completed,
  t.distance_km,
  t.duration_min,
  t.surge_multiplier,
  t.fare_inr,
  t.rider_rating,
  (unix_timestamp(t.pickup_ts) - unix_timestamp(t.request_ts)) / 60.0 AS wait_min,
  p.payment_method,
  p.payment_status
FROM rideflow.silver.trips t
LEFT JOIN rideflow.silver.dim_driver d
  ON  t.driver_id  = d.driver_id
  AND t.request_ts >= d.effective_from
  AND t.request_ts <  COALESCE(d.effective_to, TIMESTAMP '9999-12-31')
JOIN rideflow.silver.cities c
  ON t.city_id = c.city_id
LEFT JOIN rideflow.silver.payments p
  ON t.trip_id = p.trip_id AND NOT p.is_orphan

In [0]:
%sql
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT trip_id) AS distinct_trips,
       SUM(CASE WHEN driver_id IS NOT NULL AND driver_sk IS NULL THEN 1 ELSE 0 END) AS missing_driver_version
FROM rideflow.gold.fact_trips

In [0]:
%sql
CREATE OR REPLACE TABLE rideflow.gold.daily_city_kpis AS
SELECT
  trip_date_ist,
  city_id,
  city_name,
  COUNT(*)                                                             AS total_requests,
  SUM(CASE WHEN is_completed THEN 1 ELSE 0 END)                        AS completed_trips,
  ROUND(100.0 * SUM(CASE WHEN NOT is_completed THEN 1 ELSE 0 END) / COUNT(*), 1) AS cancellation_rate_pct,
  ROUND(SUM(CASE WHEN is_completed THEN fare_inr ELSE 0 END), 2)       AS gross_revenue_inr,
  ROUND(AVG(CASE WHEN is_completed THEN fare_inr END), 2)              AS avg_fare_inr,
  ROUND(AVG(CASE WHEN is_completed THEN distance_km END), 2)           AS avg_distance_km,
  ROUND(AVG(wait_min), 1)                                              AS avg_wait_min,
  ROUND(100.0 * SUM(CASE WHEN surge_multiplier > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS surge_trip_pct
FROM rideflow.gold.fact_trips
GROUP BY trip_date_ist, city_id, city_name

In [0]:
%sql
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT trip_date_ist) AS days,
       SUM(total_requests) AS total_requests
FROM rideflow.gold.daily_city_kpis


In [0]:
%sql
CREATE OR REPLACE TABLE rideflow.gold.driver_leaderboard AS
WITH weekly AS (
  SELECT
    date_trunc('WEEK', trip_date_ist)                         AS week_start,
    city_name,
    driver_id,
    COUNT(*)                                                  AS trips,
    SUM(CASE WHEN is_completed THEN fare_inr ELSE 0 END)      AS revenue_inr,
    ROUND(AVG(rider_rating), 2)                               AS avg_rider_rating,
    SUM(CASE WHEN trip_status = 'CANCELLED_BY_DRIVER' THEN 1 ELSE 0 END) AS driver_cancellations
  FROM rideflow.gold.fact_trips
  WHERE driver_id IS NOT NULL
  GROUP BY 1, 2, 3
)
SELECT *,
       dense_rank() OVER (PARTITION BY week_start, city_name ORDER BY revenue_inr DESC) AS rank_in_city
FROM weekly

In [0]:
%sql
SELECT * FROM rideflow.gold.driver_leaderboard
WHERE rank_in_city <= 3
ORDER BY city_name, rank_in_city

In [0]:
%sql
CREATE OR REPLACE TABLE rideflow.gold.hourly_demand AS
SELECT
  city_name,
  hour(from_utc_timestamp(request_ts, 'Asia/Kolkata'))                AS hour_ist,
  COUNT(*)                                                             AS requests,
  ROUND(100.0 * SUM(CASE WHEN trip_status='COMPLETED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS completion_rate_pct
FROM rideflow.gold.fact_trips
GROUP BY city_name,hour_ist

In [0]:
%sql
SELECT COUNT(*) AS rows, SUM(requests) AS total FROM rideflow.gold.hourly_demand

In [0]:
%sql
SELECT hour_ist, SUM(requests) AS requests
FROM rideflow.gold.hourly_demand GROUP BY 1 ORDER BY 1

In [0]:
%sql
CREATE OR REPLACE TABLE rideflow.gold.payment_mix AS
SELECT
  trip_date_ist,
  payment_method,
  COUNT(*)                                                                    AS payments,
  ROUND(SUM(fare_inr), 2)                                                     AS amount_inr,
  ROUND(100.0 * SUM(CASE WHEN payment_status = 'FAILED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_rate_pct
FROM rideflow.gold.fact_trips
WHERE payment_method IS NOT NULL
GROUP BY trip_date_ist, payment_method

In [0]:
%sql
SELECT COUNT(*) AS rows, SUM(payments) AS total_payments
FROM rideflow.gold.payment_mix

In [0]:
%sql
SELECT
  COUNT(*)                                                    AS all_payments,
  SUM(CASE WHEN p.is_orphan THEN 1 ELSE 0 END)                AS orphans,
  SUM(CASE WHEN q.trip_id IS NOT NULL THEN 1 ELSE 0 END)      AS for_quarantined_trips
FROM rideflow.silver.payments p
LEFT JOIN (SELECT DISTINCT trip_id FROM rideflow.silver.trips_quarantine) q
  ON p.trip_id = q.trip_id

In [0]:
%sql
-- 1. Compact and cluster the files
OPTIMIZE rideflow.gold.fact_trips;

-- 2. Document the tables (the dashboard and Genie read these descriptions)
COMMENT ON TABLE rideflow.gold.daily_city_kpis IS 'One row per IST date per city. Main table for the RideFlow Ops dashboard.';
ALTER TABLE rideflow.gold.daily_city_kpis ALTER COLUMN cancellation_rate_pct COMMENT 'Non-completed requests / all requests x 100';

In [0]:
%sql
SELECT 'bronze.trips_raw'  AS t, COUNT(*) AS n FROM rideflow.bronze.trips_raw
UNION ALL SELECT 'bronze.payments_raw',  COUNT(*) FROM rideflow.bronze.payments_raw
UNION ALL SELECT 'bronze.drivers_cdc_raw', COUNT(*) FROM rideflow.bronze.drivers_cdc_raw
UNION ALL SELECT 'silver.trips',     COUNT(*) FROM rideflow.silver.trips
UNION ALL SELECT 'silver.quarantine (distinct)', COUNT(DISTINCT trip_id) FROM rideflow.silver.trips_quarantine
UNION ALL SELECT 'silver.payments orphans', SUM(CASE WHEN is_orphan THEN 1 ELSE 0 END) FROM rideflow.silver.payments

In [0]:
%sql
SELECT COUNT(*) FROM rideflow.silver.trips_quarantine


In [0]:
%sql
SELECT driver_id, driver_sk, city_id, rating, status, effective_from, effective_to, is_current
FROM rideflow.silver.dim_driver
WHERE driver_id IN (SELECT driver_id FROM rideflow.silver.dim_driver GROUP BY driver_id HAVING COUNT(*) > 1)
ORDER BY driver_id, effective_from
LIMIT 20

In [0]:
%sql
SELECT 'bronze.trips_raw' AS t, COUNT(*) AS n FROM rideflow.bronze.trips_raw
UNION ALL SELECT 'bronze.payments_raw', COUNT(*) FROM rideflow.bronze.payments_raw
UNION ALL SELECT 'bronze.drivers_cdc_raw', COUNT(*) FROM rideflow.bronze.drivers_cdc_raw
UNION ALL SELECT 'silver.trips', COUNT(*) FROM rideflow.silver.trips
UNION ALL SELECT 'silver.quarantine (distinct)', COUNT(DISTINCT trip_id) FROM rideflow.silver.trips_quarantine
UNION ALL SELECT 'dim_driver total', COUNT(*) FROM rideflow.silver.dim_driver
UNION ALL SELECT 'dim_driver current', SUM(CASE WHEN is_current THEN 1 ELSE 0 END) FROM rideflow.silver.dim_driver
UNION ALL SELECT 'gold.fact_trips', COUNT(*) FROM rideflow.gold.fact_trips
UNION ALL SELECT 'gold kpi days', COUNT(DISTINCT trip_date_ist) FROM rideflow.gold.daily_city_kpis

In [0]:
%sql
SELECT trip_date_ist, cancellation_rate_pct
FROM rideflow.gold.daily_city_kpis
WHERE city_name = 'Mumbai' ORDER BY trip_date_ist